In [38]:
import sqlite3
import pandas as pd

df = pd.read_csv("/content/student_dataset_1000.csv")
df.head()

,student_id,name,department,math,science,programming,attendance
0,1,Sumit Sharma,CS,76,66,50,83
1,2,Divya Singh,IT,79,76,76,98
2,3,Amit Verma,ECE,72,71,68,93
3,4,Karan Nair,IT,66,58,49,72
4,5,Tarun Singh,ECE,82,83,79,83


##Create SQLite Database & Table

In [39]:
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()


df.to_sql('students', conn, index=False, if_exists='replace')

1000

##Display Student Records

In [40]:
query = "SELECT * FROM students LIMIT 6;"
print(pd.read_sql(query, conn))

   student_id          name department  math  science  programming  attendance
0           1  Sumit Sharma         CS    76       66           50          83
1           2   Divya Singh         IT    79       76           76          98
2           3    Amit Verma        ECE    72       71           68          93
3           4    Karan Nair         IT    66       58           49          72
4           5   Tarun Singh        ECE    82       83           79          83
5           6   Kavya Verma         IT    90       99           86          81


##Average Marks of Each Student

In [41]:
query = """
SELECT student_id, name,
       (math + science + programming)/3.0 AS avg_marks
FROM students;
"""
print(pd.read_sql(query, conn))

     student_id           name  avg_marks
0             1   Sumit Sharma  64.000000
1             2    Divya Singh  77.000000
2             3     Amit Verma  70.333333
3             4     Karan Nair  57.666667
4             5    Tarun Singh  81.333333
..          ...            ...        ...
995         996    Swati Reddy  69.666667
996         997    Tarun Kumar  70.666667
997         998   Vikram Reddy  70.333333
998         999  Deepika Patel  57.000000
999        1000     Amit Verma  75.333333

[1000 rows x 3 columns]


##Top-Performing Student

In [42]:
query = """
SELECT student_id, name,
       (math + science + programming)/3.0 AS avg_marks
FROM students
ORDER BY avg_marks DESC
LIMIT 1;
"""
print(pd.read_sql(query, conn))

   student_id          name  avg_marks
0         176  Pallavi Nair      100.0


##Department-wise Average Marks

In [43]:
query = """
SELECT department,
       AVG((math + science + programming)/3.0) AS dept_avg
FROM students
GROUP BY department;
"""
print(pd.read_sql(query, conn))

  department   dept_avg
0         CS  71.575461
1      Civil  72.877193
2        ECE  70.882263
3        EEE  70.541667
4         IT  71.344569
5         ME  71.157895


## Students Scoring Below 60 (in any subject)

In [44]:
query = """
SELECT * FROM students
WHERE math < 60 OR science < 60 OR programming < 60;
"""
print(pd.read_sql(query, conn))

     student_id           name department  math  science  programming  \
0             1   Sumit Sharma         CS    76       66           50   
1             4     Karan Nair         IT    66       58           49   
2             7   Suresh Singh         CS    62       36           44   
3            11    Varun Reddy        ECE    64       75           54   
4            12    Smita Gupta         CS    51       67           62   
..          ...            ...        ...   ...      ...          ...   
369         990     Divya Nair         IT    58       70           73   
370         994   Rekha Sharma         IT    39       38           31   
371         995   Vivek Sharma        ECE    51       62           51   
372         997    Tarun Kumar      Civil    82       80           50   
373         999  Deepika Patel        ECE    54       60           57   

     attendance  
0            83  
1            72  
2            86  
3            97  
4            97  
..          ...

##Overall Class Average

In [45]:
query = """
SELECT AVG((math + science + programming)/3.0) AS overall_avg
FROM students;
"""
print(pd.read_sql(query, conn))

   overall_avg
0       71.372


##Count Students in Each Department

In [46]:
query = """
SELECT department, COUNT(*) AS total_students
FROM students
GROUP BY department;
"""
print(pd.read_sql(query, conn))

  department  total_students
0         CS             307
1      Civil              76
2        ECE             218
3        EEE              56
4         IT             267
5         ME              76


##Find Students with Attendance > 90

In [47]:
query = """
SELECT student_id, name, attendance
FROM students
WHERE attendance > 90;
"""
print(pd.read_sql(query, conn))

     student_id          name  attendance
0             2   Divya Singh          98
1             3    Amit Verma          93
2             8   Karan Gupta          96
3            11   Varun Reddy          97
4            12   Smita Gupta          97
..          ...           ...         ...
279         985   Gaurav Nair         100
280         990    Divya Nair          91
281         993   Vivek Gupta          91
282         998  Vikram Reddy          98
283        1000    Amit Verma         100

[284 rows x 3 columns]


##Find Highest Marks in Each Subject

In [48]:
query = """
SELECT
    MAX(math) AS max_math,
    MAX(science) AS max_science,
    MAX(programming) AS max_programming
FROM students;
"""
print(pd.read_sql(query, conn))

   max_math  max_science  max_programming
0       100          100              100


##Students Above Class Average

In [49]:
query = """
SELECT student_id, name,
       (math + science + programming)/3.0 AS avg_marks
FROM students
WHERE (math + science + programming)/3.0 >
      (SELECT AVG((math + science + programming)/3.0) FROM students);
"""
print(pd.read_sql(query, conn))

     student_id           name  avg_marks
0             2    Divya Singh  77.000000
1             5    Tarun Singh  81.333333
2             6    Kavya Verma  91.666667
3             8    Karan Gupta  73.000000
4             9   Ruchika Nair  85.333333
..          ...            ...        ...
500         989  Deepika Singh  86.666667
501         991  Siddharth Das  84.666667
502         992  Ruchika Joshi  82.333333
503         993    Vivek Gupta  73.333333
504        1000     Amit Verma  75.333333

[505 rows x 3 columns]
